# NDR - Notebook 37: Repair = L_OD(clean) + L_SUP(patch) + EWC (fix s_diff)

## Bai hoc that (LB)
- NB36: forget trigger 10/15->0/15 NHUNG calibration drift -> s_diff demote 88% / soft 5.8% -> LB 365.
- Nguyen nhan: chi L_SUP patch + L_PRES score yeu, **thieu full detection loss tren clean**.
- Paper 2605.05928: L_OD(x)+L_OD(x') + defense localize. Prune collapse.
- 229: s_diff EWC demote manh + conf goc poisoned = LB 229.

## Design NB37
1. Build base 229-style: prune 76ch + cls-only EWC (lambda500) -> weights co calibration on hon poisoned raw.
2. Repair head (backbone freeze):
   - **L_OD clean**: Detectron2 detection loss tren anh (khong patch) + GT = streak-like pseudo-label.
   - **L_SUP patch**: dan patch that, chi ep sigmoid^2 tren region mask patch (khong dilute).
   - **EWC** neo ve base 229 weights.
3. Verify (bat buoc truoc rescore):
   - firing patch giam.
   - s_diff mean tren poison-transplant **>** s_diff mean tren high-conf streak-like + margin.
   - SEP_OK flag.
4. Neu SEP_OK: rescore 229 (candidate poisoned + s_diff NB37 + geo), P_HI/P_LO quantile demote~30%.
5. Neu khong SEP_OK: **khong nop repair** — ghi submission = 229-style rescore voi s_diff tu base EWC (anchor).
Xuat 1 file submission.csv.


## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 14.4 MB/s eta 0:00:00


In [2]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


Setup OK | device: cuda | EWC sweep: [500.0, 1000.0, 2000.0, 4000.0]


## 2. Data + helpers (229 exact)

In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


unlearn: 20 | test: 2000


## 3. maCADD metric + iou

In [4]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


metric ready


## 4. Base 229-style: prune + cls-only EWC (calibration tot hon poisoned raw)


In [5]:
# --- prune activation-guided (229) ---
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target=[m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target):
        stored[i]=[]
        hooks.append(layer.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc="acts"):
            im=load_image(d["file_name"])
            model([{"image": torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    spl={}
    for li, acts in stored.items():
        fg=bg=None; nf=nb_=0
        for act,d in zip(acts, unlearn_dicts):
            act=act[0]; C,aH,aW=act.shape
            sy,sx=aH/d["height"], aW/d["width"]; boxes=poison_boxes.get(d["image_id"],[])
            if fg is None: fg=torch.zeros(C); bg=torch.zeros(C)
            for x,y,w,h in boxes:
                x1=max(0,int(x*sx)); y1=max(0,int(y*sy)); x2=min(aW,int((x+w)*sx)+1); y2=min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg+=act[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            rng=np.random.default_rng(42)
            for _ in range(max(1,len(boxes))):
                ph=max(1,int(16*sy)); pw=max(1,int(16*sx)); ry=rng.integers(0,max(1,aH-ph)); rx=rng.integers(0,max(1,aW-pw))
                bg+=act[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        spl[li]=(fg-bg).numpy()
    return spl

def prune_channels(model, spl, prune_frac=0.15):
    target=[(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    tot=0
    for i,layer in target:
        if i not in spl: continue
        sc=spl[i]; n_pr=max(1,int(len(sc)*prune_frac)); bad=np.argsort(sc)[-n_pr:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        tot+=n_pr
    print("pruned", tot, "channels"); return model

class ClsEWCTrainer(DefaultTrainer):
    anchor_weights=None; ewc_lambda=500.0
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=UInt16DatasetMapper(cfg, is_train=True, augmentations=[]),
                                            dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
    @classmethod
    def build_model(cls, cfg):
        m=super().build_model(cfg)
        for n,p in m.named_parameters():
            p.requires_grad=("cls_score" in n or "cls_subnet" in n)
        return m
    def run_step(self):
        assert self.model.training
        if not hasattr(self,"_dli"): self._dli=iter(self.data_loader)
        try: data=next(self._dli)
        except StopIteration: self._dli=iter(self.data_loader); data=next(self._dli)
        ld=self.model(data)
        if self.anchor_weights is not None:
            ewc=torch.tensor(0.0, device=DEVICE)
            for n,p in self.model.named_parameters():
                if p.requires_grad and n in self.anchor_weights:
                    ewc=ewc+((p-self.anchor_weights[n])**2).sum()
            ld["loss_ewc"]=self.ewc_lambda*ewc
        loss=sum(ld.values()); self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()

cfg=build_cfg(); model=build_model(cfg); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model=model.to(DEVICE)
stored=collect_activations(model, unlearn_dicts, poison_boxes)
spl=compute_poison_scores(stored, unlearn_dicts, poison_boxes)
model=prune_channels(model, spl, 0.15)
PRUNED=str(WORK/"pruned37.pth"); torch.save({"model":model.state_dict()}, PRUNED)
ANCHOR_EWC={n:p.detach().clone() for n,p in model.named_parameters() if ("cls_score" in n or "cls_subnet" in n)}
del model; torch.cuda.empty_cache()

out=str(WORK/"base229"); Path(out).mkdir(parents=True, exist_ok=True)
cfg=build_cfg(weights=PRUNED); cfg.DATASETS.TRAIN=(UNLEARN_DATASET,); cfg.DATASETS.TEST=()
cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE
cfg.SOLVER.BASE_LR=2.5e-4; cfg.SOLVER.MAX_ITER=20; cfg.SOLVER.STEPS=[]; cfg.OUTPUT_DIR=out
ClsEWCTrainer.anchor_weights=ANCHOR_EWC; ClsEWCTrainer.ewc_lambda=500.0
tr=ClsEWCTrainer(cfg); tr.resume_or_load(resume=False); tr.train()
BASE229=str(Path(out)/"model_final.pth")
print("base229 ready ->", BASE229)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
acts:   0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
acts: 100%|██████████| 20/20 [00:05<00:00,  3.51it/s]
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


pruned 76 channels
[07/17 09:52:33 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
 

2026-07-17 09:52:47.528901: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784281967.734458      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784281967.791024      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784281968.274432      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784281968.274460      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784281968.274463      23 computation_placer.cc:177] computation placer alr

[07/17 09:52:59 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:08 (0.4871 s / it)
[07/17 09:52:59 d2.engine.hooks]: Total training time: 0:00:24 (0:00:15 on hooks)
base229 ready -> /kaggle/working/base229/model_final.pth


## 5. Poison patches THAT + streak-like retain (pseudo-GT)


In [6]:
from detectron2.structures import BoxMode, Boxes, Instances
from detectron2.utils.events import EventStorage

poisoned_pred=build_predictor(POISONED_WEIGHTS, CAND_THRESH)
base_pred=build_predictor(BASE229, CAND_THRESH)
def pr(predictor, img):
    inst=predictor(img)["instances"].to("cpu")
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

stem_by_id={im["id"]: str(int(Path(im["file_name"]).stem)) for im in coco_data["images"]}
unlearn_imgs={str(int(p.stem)): load_image(p) for p in unlearn_files}
MARGIN=6; poison_patches=[]
for a in coco_data["annotations"]:
    x,y,w,h=a["bbox"]; st=stem_by_id[a["image_id"]]; img=unlearn_imgs.get(st)
    if img is None: continue
    x0,y0=int(max(0,x-MARGIN)),int(max(0,y-MARGIN)); x1,y1=int(min(IMG_W,x+w+MARGIN)),int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(img[y0:y1,x0:x1].copy())
print("poison patches:", len(poison_patches))

def is_streak_like(box):
    x1,y1,x2,y2=map(float,box); w=max(1.0,x2-x1); h=max(1.0,y2-y1); ar=max(w/h,h/w); area=w*h
    return (ar>=2.2) and (8<=max(w,h)<=180) and (area<=6000)

def synth_streak_on(bg,rng):
    h,w=bg.shape; canvas=np.zeros_like(bg)
    ang=rng.uniform(0,np.pi); L=rng.uniform(20,110); cx,cy=rng.uniform(0.25,0.75)*w,rng.uniform(0.25,0.75)*h
    p1=(int(cx-L/2*np.cos(ang)),int(cy-L/2*np.sin(ang))); p2=(int(cx+L/2*np.cos(ang)),int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas=cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp=rng.uniform(3,14)*(np.std(bg)+1e-3); out=bg+canvas/(canvas.max()+1e-6)*amp
    box=[min(p1[0],p2[0])-3,min(p1[1],p2[1])-3,max(p1[0],p2[0])+3,max(p1[1],p2[1])+3]
    return out,[float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]

RETAIN_CONF=0.4; IMG_STORE={}
def build_retain(n_test=250, seed=SEED):
    rng=np.random.default_rng(seed); records=[]
    pick=list(range(len(test_files))); rng.shuffle(pick); pick=pick[:n_test]
    for k in pick:
        p=test_files[k]; img=load_image(p); streak=[]
        b,sc=pr(poisoned_pred,img); rb=b[sc>=RETAIN_CONF] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            box=[float(x1),float(y1),float(x2),float(y2)]
            if x2-x1>1 and y2-y1>1 and is_streak_like(box): streak.append(box)
        g=img[:,:,0].copy()
        for _ in range(rng.integers(1,4)):
            g,box=synth_streak_on(g,rng); x1,y1,x2,y2=box
            if x2-x1>1 and y2-y1>1: streak.append([float(x1),float(y1),float(x2),float(y2)])
        img=np.repeat(g[:,:,None],3,axis=2)
        iid=20_000_000+int(k); IMG_STORE[iid]=img.astype(np.float32)
        records.append({"image_id":iid,"streak_boxes":streak})
    return records
RECORDS=build_retain(250)
print("retain", len(RECORDS), "avg streak", round(np.mean([len(r["streak_boxes"]) for r in RECORDS]),2))


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/17 09:53:00 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/17 09:53:01 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/base229/model_final.pth ...
poison patches: 20
retain 250 avg streak 2.34


## 6. Repair: L_OD(clean GT) + L_SUP(patch mask) + EWC(base229)
- Clean path: anh **khong** patch, GT instances = streak boxes -> full RetinaNet loss (paper L_OD).
- Patch path: dan patch that, chi L_SUP localize (sigmoid^2 tren mask) — khong dilute.
- Freeze backbone; EWC neo ve BASE229.


In [7]:
def make_instances(streak_boxes, H=IMG_H, W=IMG_W, device=DEVICE):
    inst=Instances((H,W))
    if len(streak_boxes)==0:
        inst.gt_boxes=Boxes(torch.zeros((0,4), device=device))
        inst.gt_classes=torch.zeros((0,), dtype=torch.int64, device=device)
    else:
        xyxy=torch.tensor(streak_boxes, dtype=torch.float32, device=device)
        # clip
        xyxy[:,0::2].clamp_(0,W); xyxy[:,1::2].clamp_(0,H)
        inst.gt_boxes=Boxes(xyxy)
        inst.gt_classes=torch.zeros((len(streak_boxes),), dtype=torch.int64, device=device)
    return inst

def cls_logits(model, x255):
    nrm=((x255-model.pixel_mean)/model.pixel_std)[None]
    feats=model.backbone(nrm); fl=[feats[f] for f in model.head_in_features]
    logits,_=model.head(fl)
    return logits

def region_mask(logit, boxes, H, W):
    _,_,Hf,Wf=logit.shape; m=torch.zeros(1,1,Hf,Wf, device=logit.device)
    sy,sx=Hf/H, Wf/W
    for (x1,y1,x2,y2) in boxes:
        a=int(y1*sy); b=max(int(y2*sy)+1,a+1); c=int(x1*sx); d=max(int(x2*sx)+1,c+1)
        m[:,:,a:b,c:d]=1.0
    return m

def repair37(n_test, lr, iters, ewc_lambda, lam_sup, drop_p, seed, tag):
    out=WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model=build_model(build_cfg(BASE229)); DetectionCheckpointer(model).load(BASE229); model.to(DEVICE)
    for n,p in model.named_parameters():
        p.requires_grad = not n.startswith("backbone.")
    anchor={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    opt=torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    recs=build_retain(n_test=n_test, seed=seed)
    rng=np.random.default_rng(seed)
    model.train()
    with EventStorage(0):
        for step in range(iters):
            r=recs[int(rng.integers(len(recs)))]
            img0=IMG_STORE[r["image_id"]].copy()
            streak=[list(map(float,b)) for b in r["streak_boxes"]]
            # ---- clean L_OD ----
            xt=torch.as_tensor(img0.transpose(2,0,1).copy(), dtype=torch.float32, device=DEVICE)
            inst=make_instances(streak)
            data=[{"image": xt, "instances": inst, "height": IMG_H, "width": IMG_W}]
            ld=model(data)
            loss=sum(ld.values())
            # ---- patch L_SUP (optional) ----
            if rng.random()<drop_p and len(poison_patches):
                img1=img0.copy(); patch_boxes=[]
                for _ in range(int(rng.integers(2,5))):
                    pt=poison_patches[int(rng.integers(len(poison_patches)))]; ph,pw=pt.shape[:2]
                    if ph>=IMG_H or pw>=IMG_W: continue
                    y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
                    img1[y0:y0+ph,x0:x0+pw]=pt; patch_boxes.append([x0,y0,x0+pw,y0+ph])
                if patch_boxes:
                    xt1=torch.as_tensor(img1.transpose(2,0,1).copy(), dtype=torch.float32, device=DEVICE)
                    # still need train mode for BN-ish consistency; use logits path
                    logits=cls_logits(model, xt1)
                    lsup=torch.tensor(0.0, device=DEVICE)
                    for lg in logits:
                        s=torch.sigmoid(lg); m=region_mask(lg, patch_boxes, IMG_H, IMG_W)
                        lsup=lsup+((s**2)*m).sum()/(m.sum()+1e-6)
                    loss=loss+lam_sup*lsup
            # EWC
            ewc=torch.tensor(0.0, device=DEVICE)
            for n,p in model.named_parameters():
                if p.requires_grad and n in anchor:
                    ewc=ewc+((p-anchor[n])**2).sum()
            loss=loss+ewc_lambda*ewc
            opt.zero_grad(); loss.backward(); opt.step()
            if (step+1)%50==0:
                print(f"  {tag} it{step+1}: loss={float(loss.detach()):.4f}")
    final=str(out/"model_final.pth"); torch.save({"model": model.state_dict()}, final)
    del model; torch.cuda.empty_cache()
    return final

print("repair37 ready")


repair37 ready


## 7. Train + VERIFY (firing + s_diff separation). Rescore chi khi SEP_OK.
SEP: s_diff mean(poison transplant) - s_diff mean(high-conf streak-like) >= 0.15


In [8]:
REP=repair37(n_test=250, lr=5e-5, iters=300, ewc_lambda=5e-4, lam_sup=2.5, drop_p=0.9, seed=42, tag="rep37b")
print("repaired ->", REP)
rep_pred=build_predictor(REP, CAND_THRESH)

def s_diff_vec(cb,cs,db,ds):
    sd=np.ones(len(cb),np.float32)
    if len(cb) and len(db):
        ious=iou_matrix(cb,db); j=ious.argmax(1); bi=ious[np.arange(len(cb)),j]
        for i in range(len(cb)):
            if bi[i]>=0.5: sd[i]=1.0-(ds[j[i]]/max(cs[i],1e-6))
    return np.clip(sd,0,1)

# verify firing on real patches
def verify_firing(n=30, seed=1):
    rng=np.random.default_rng(seed); on_p=on_r=0; ntot=0
    for k in rng.choice(len(test_files), size=n, replace=False):
        img=load_image(test_files[int(k)]); pt=poison_patches[int(rng.integers(len(poison_patches)))]; ph,pw=pt.shape[:2]
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im2=img.copy(); im2[y0:y0+ph,x0:x0+pw]=pt; box=np.array([[x0,y0,x0+pw,y0+ph]],np.float32)
        bp,sp=pr(poisoned_pred,im2); br,sr=pr(rep_pred,im2)
        fp=len(bp) and iou_matrix(box,bp).max()>=0.1 and sp[iou_matrix(box,bp)[0].argmax()]>0.2
        fr=len(br) and iou_matrix(box,br).max()>=0.1 and sr[iou_matrix(box,br)[0].argmax()]>0.2
        on_p+=int(bool(fp)); on_r+=int(bool(fr)); ntot+=1
    print(f"firing patch: POISONED={on_p}/{ntot} -> REPAIRED={on_r}/{ntot}")
    return on_p/ntot, on_r/ntot

# s_diff separation: poison boxes vs streak-like high conf
def verify_sep(n_img=120, seed=2):
    rng=np.random.default_rng(seed)
    sd_poison=[]; sd_real=[]
    for k in rng.choice(len(test_files), size=n_img, replace=False):
        img=load_image(test_files[int(k)])
        # real: high conf streak-like from poisoned
        cb,cs=pr(poisoned_pred,img); db,ds=pr(rep_pred,img)
        if len(cb):
            sd=s_diff_vec(cb,cs,db,ds)
            for i in range(len(cb)):
                if cs[i]>=0.35 and is_streak_like(cb[i]):
                    sd_real.append(float(sd[i]))
        # poison: paste patch then take detections overlapping patch
        pt=poison_patches[int(rng.integers(len(poison_patches)))]; ph,pw=pt.shape[:2]
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im2=img.copy(); im2[y0:y0+ph,x0:x0+pw]=pt; box=np.array([[x0,y0,x0+pw,y0+ph]],np.float32)
        cb2,cs2=pr(poisoned_pred,im2); db2,ds2=pr(rep_pred,im2)
        if len(cb2):
            sd2=s_diff_vec(cb2,cs2,db2,ds2)
            ious=iou_matrix(cb2, box)
            for i in range(len(cb2)):
                if ious[i,0]>=0.1 and cs2[i]>=0.2:
                    sd_poison.append(float(sd2[i]))
    mp=float(np.mean(sd_poison)) if sd_poison else 0.0
    mr=float(np.mean(sd_real)) if sd_real else 1.0
    sep=mp-mr
    print(f"s_diff mean poison_overlap={mp:.3f} (n={len(sd_poison)}) | streak_like={mr:.3f} (n={len(sd_real)}) | SEP={sep:.3f}")
    return mp, mr, sep, len(sd_poison), len(sd_real)

fp,fr=verify_firing()
mp,mr,sep,np_,nr=verify_sep()
FIRE_OK = (fp-fr) >= 0.35 and fr <= 0.25
SEP_OK = (sep >= 0.15) and (np_ >= 20) and (nr >= 40)
print(f"targets: fire_repair<=0.25, delta_fire>=0.35, sep>=0.15, n_real>=40")
print(f"FIRE_OK={FIRE_OK} SEP_OK={SEP_OK} -> USE_REPAIR={FIRE_OK and SEP_OK}")
USE_REPAIR = bool(FIRE_OK and SEP_OK)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/17 09:53:36 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/base229/model_final.pth ...
  rep37b it50: loss=0.2433
  rep37b it100: loss=0.5197
  rep37b it150: loss=0.2741
  rep37b it200: loss=0.3845
  rep37b it250: loss=0.5916
  rep37b it300: loss=0.5007


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


repaired -> /kaggle/working/rep37b/model_final.pth
[07/17 09:55:33 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/rep37b/model_final.pth ...
firing patch: POISONED=16/30 -> REPAIRED=5/30
s_diff mean poison_overlap=0.742 (n=140) | streak_like=0.568 (n=45) | SEP=0.174
targets: fire_repair<=0.25, delta_fire>=0.35, sep>=0.15, n_real>=40
FIRE_OK=True SEP_OK=True -> USE_REPAIR=True


## 8. Rescore 229-style. Neu !USE_REPAIR -> s_diff tu base229 (anchor an toan). 1 file.


In [9]:
# geometry + rescore
logwh=np.log(np.array([ann["bbox"][2:4] for ann in coco_data["annotations"]],np.float32))
MU=logwh.mean(0); COV=np.cov(logwh.T)+1e-3*np.eye(2); COV_INV=np.linalg.inv(COV)
def geometry_scores(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    wh=np.stack([boxes[:,2]-boxes[:,0],boxes[:,3]-boxes[:,1]],1)
    d=np.log(np.maximum(wh,1e-3))-MU; m2=np.einsum("ni,ij,nj->n",d,COV_INV,d)
    return np.exp(-0.5*m2/2.0).astype(np.float32)

MIN_KEEP=0.20; EPS=0.01; W_DIFF,W_GEO=0.90,0.10
TARGET_DEMOTE_FRAC=0.30; KEEP_FRAC=0.40

# choose depoison predictor for s_diff
dep_for_sd = rep_pred if USE_REPAIR else base_pred
tag = "REPAIR37" if USE_REPAIR else "BASE229_ANCHOR"
print("using s_diff from:", tag)

CACHE={}; all_p=[]
for pth in tqdm(test_files, desc=f"cache cand+s_diff[{tag}]"):
    img=load_image(pth)
    cb,cs=pr(poisoned_pred,img); db,ds=pr(dep_for_sd,img)
    sd=s_diff_vec(cb,cs,db,ds); sg=geometry_scores(cb)
    pp=np.clip(W_DIFF*sd+W_GEO*sg,0,1)
    CACHE[pth.stem]=(cb,cs,pp)
    if len(cs): all_p.extend(pp[cs>=MIN_KEEP].tolist())
all_p=np.array(all_p,np.float32)
if len(all_p)==0:
    P_LO,P_HI=0.25,0.55
else:
    P_HI=float(np.quantile(all_p, 1.0-TARGET_DEMOTE_FRAC))
    P_LO=float(np.quantile(all_p, KEEP_FRAC))
    if P_LO>=P_HI:
        P_LO,P_HI=float(np.quantile(all_p,0.3)), float(np.quantile(all_p,0.7))
print(f"P_LO={P_LO:.4f} P_HI={P_HI:.4f} | p med={np.median(all_p):.3f} max={all_p.max():.3f}")

def remap(s,p):
    if s<MIN_KEEP: return 0.0
    if p>=P_HI: return EPS
    if p<=P_LO: return float(s)
    frac=(p-P_LO)/max(P_HI-P_LO,1e-6)
    return float(max(EPS,s*(1-frac)))

sample=pd.read_csv(SAMPLE_SUB,dtype={"image_id":str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f"{float(sc):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"]
    return " ".join(parts) if parts else " "

pred={}; n_eps=n_keep=n_ramp=0; before=after=0.0
for stem,(cb,cs,pp) in CACHE.items():
    if len(cb)==0:
        pred[stem]=(cb,cs); continue
    new=np.array([remap(float(s),float(p)) for s,p in zip(cs,pp)],np.float32)
    for s,v in zip(cs,new):
        if s>=MIN_KEEP:
            before+=float(s)
            if v<=EPS+1e-6: n_eps+=1
            elif abs(v-float(s))<1e-6: n_keep+=1
            else: n_ramp+=1
    keep=new>0.0
    eps_ids=np.where(new<=EPS+1e-6)[0]; strong=np.where(new>0.20)[0]
    if len(eps_ids) and len(strong):
        ov=iou_matrix(cb[eps_ids],cb[strong]).max(1); keep[eps_ids[ov>=0.20]]=False
    b,s=cb[keep],new[keep]; after+=float(s.sum()); pred[stem]=(b,s)

sub=sample.copy()
sub["prediction_string"]=sub["image_id"].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
sub["prediction_string"]=sub["prediction_string"].fillna(" ")
sub.loc[sub["prediction_string"].str.strip()=="","prediction_string"]=" "
sub.to_csv(str(WORK/"submission.csv"),index=False)
tot=sum(len(s) for _,s in pred.values())
print(f"saved submission.csv | source={tag} | dets={tot} ({tot/len(test_files):.3f}/img)")
print(f"eps={n_eps} ramp={n_ramp} keep={n_keep} | demote_rate={n_eps/max(n_eps+n_ramp+n_keep,1):.1%}")
print(f"conf_mass before={before:.1f} after={after:.1f} drop={before-after:.1f}")
print("If USE_REPAIR=False this is essentially 229-class rescore (safe). LB vs 229 is the only judge.")


using s_diff from: REPAIR37


cache cand+s_diff[REPAIR37]: 100%|██████████| 2000/2000 [06:41<00:00,  4.99it/s]


P_LO=0.6759 P_HI=0.8301 | p med=0.735 max=1.000
saved submission.csv | source=REPAIR37 | dets=2470 (1.235/img)
eps=790 ramp=766 keep=1037 | demote_rate=30.5%
conf_mass before=1401.5 after=842.3 drop=559.1
If USE_REPAIR=False this is essentially 229-class rescore (safe). LB vs 229 is the only judge.
